In [46]:
import pyautogui as pyg
pyg.sleep(2)
basex=412
basey=489
gap=50

In [47]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)


0 0 412 489 (24, 16, 20)
0 1 462 489 (25, 17, 20)
0 2 512 489 (26, 17, 20)
0 3 562 489 (27, 18, 20)
0 4 612 489 (27, 18, 20)
0 5 662 489 (28, 19, 21)
0 6 712 489 (28, 19, 21)
0 7 762 489 (31, 21, 22)
1 0 412 539 (34, 24, 23)
1 1 462 539 (33, 24, 22)
1 2 512 539 (33, 24, 23)
1 3 562 539 (33, 23, 22)
1 4 612 539 (33, 23, 22)
1 5 662 539 (34, 23, 23)
1 6 712 539 (34, 24, 23)
1 7 762 539 (35, 24, 23)
2 0 412 589 (37, 27, 26)
2 1 462 589 (37, 27, 26)
2 2 512 589 (37, 28, 28)
2 3 562 589 (37, 28, 27)
2 4 612 589 (36, 25, 24)
2 5 662 589 (37, 27, 25)
2 6 712 589 (37, 28, 25)
2 7 762 589 (37, 28, 26)
3 0 412 639 (37, 29, 27)
3 1 462 639 (37, 29, 28)
3 2 512 639 (37, 28, 28)
3 3 562 639 (37, 28, 27)
3 4 612 639 (37, 28, 27)
3 5 662 639 (37, 29, 27)
3 6 712 639 (37, 27, 28)
3 7 762 639 (37, 27, 28)
4 0 412 689 (37, 27, 25)
4 1 462 689 (37, 28, 25)
4 2 512 689 (37, 27, 27)
4 3 562 689 (37, 27, 26)
4 4 612 689 (37, 27, 26)
4 5 662 689 (37, 27, 27)
4 6 712 689 (37, 27, 27)
4 7 762 689 (37, 26, 26)


In [48]:
def get_gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [49]:
def get_gem_color(r, c, im):
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [50]:
def can_move_left(r, c, im):
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left_c = c - 1
    left_color = get_gem_color(r, left_c, im)
    current_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, left_c):
            return current_color
        if (row, col) == (r, c):
            return left_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + dr * direction
                nc = col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and color_after_swap(nr, nc) == color):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, left_c) or makes_match(r, c)

In [51]:
def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, 1, 0, im)

In [52]:
im = pyg.screenshot()
print(can_move_left(5, 4, im))

False


In [53]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))

(412, 489)
(462, 489)
(412, 539)


In [54]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, i, im))

(24, 16, 20)
(25, 17, 20)
(26, 17, 20)
(27, 18, 20)
(27, 18, 20)
(28, 19, 21)
(28, 19, 21)
(31, 21, 22)


In [55]:
import time

start_time = time.time()
directions = (
    (can_move_left, 0, -1),
    (can_move_right, 0, 1),
    (can_move_up, -1, 0),
    (can_move_down, 1, 0),
)

while time.time() - start_time < 60:
    # 每輪掃描都先取得最新棋盤畫面。
    im = pyg.screenshot()
    move_found = False

    for r in range(8):
        for c in range(8):
            if time.time() - start_time >= 60:
                break

            for can_move, dr, dc in directions:
                if time.time() - start_time >= 60:
                    break
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.2)
                    pyg.sleep(0.1)
                    # 交換後立即更新畫面，下一輪從棋盤起點重新掃描。
                    im = pyg.screenshot()
                    move_found = True
                    break

            if move_found or time.time() - start_time >= 60:
                break

        if move_found or time.time() - start_time >= 60:
            break

    if not move_found:
        # 無可消除組合時，本輪掃描結束後仍更新畫面再重試。
        im = pyg.screenshot()
        pyg.sleep(0.05)